# Note 04 Deep Dive: Span, Independence, Basis, Dimension (Applied Math for DS & AI)

Companion notebook for [`../04-Span-Independence-Basis-Dimension.md`](../04-Span-Independence-Basis-Dimension.md). Every worked example in the note is verified here.

| Part | Topic |
|---|---|
| 1 | Independence tests via rank, RREF and determinant |
| 2 | Basis of a span from pivot columns (pruning) |
| 3 | Extending an independent set to a basis |
| 4 | Coordinates in a basis and change of basis |
| 5 | Dimension formula dim(U+W) = dim U + dim W − dim(U∩W) |
| 6 | Polynomial space P₂ as coordinate vectors |
| 7 | Case study: RGB → YCbCr is a change of basis (JPEG) |
| 8 | Case study: DCT basis compression on sklearn digits |
| 9 | Case study: PCA = choosing a better basis (2-D cloud and digits) |
| 10 | Case study: one-hot vs dense embedding bases |
| 11 | Case study: the dummy-variable trap (one-hot + intercept is a dependent set) |
| 12 | Verification of the practice-problem answers (P1–P26) |

Figures for the note are produced by [`figures_04.py`](figures_04.py) (images `04x_*.png`).

In [1]:
import numpy as np
import sympy as sp
from scipy.fft import dctn, idctn
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
np.set_printoptions(precision=4, suppress=True)

def cols(*vs):
    """Matrix whose columns are the given vectors (exact, sympy)."""
    return sp.Matrix.hstack(*[sp.Matrix(v) for v in vs])

def independence_report(name, *vs):
    M = cols(*vs)
    R, piv = M.rref()
    print(f"{name}: {len(vs)} vectors in R^{M.rows}, rank = {M.rank()}, pivot columns = {piv}")
    print("  independent" if M.rank() == len(vs) else "  DEPENDENT")
    if M.rows == M.cols:
        print("  det =", M.det())
    for n in M.nullspace():
        print("  dependency (null vector c with Mc = 0):", list(n))
    return M

## Part 1: Independence tests (worked examples W1–W4)

In [2]:
independence_report("W1a", (2, 5), (1, 5))
independence_report("W1b", (1, 2), (-2, -4))
independence_report("W2 ", (1, 2, 1), (2, 1, 0), (1, -1, 2))
independence_report("W3 ", (1, 1, 0), (0, 1, 1), (1, 2, 1))
_ = independence_report("W4 ", (1, 0, 1, 2), (0, 1, 1, 1), (1, 1, 1, 0), (2, 1, 3, 5))

W1a: 2 vectors in R^2, rank = 2, pivot columns = (0, 1)
  independent
  det = 5
W1b: 2 vectors in R^2, rank = 1, pivot columns = (0,)
  DEPENDENT
  det = 0
  dependency (null vector c with Mc = 0): [2, 1]
W2 : 3 vectors in R^3, rank = 3, pivot columns = (0, 1, 2)
  independent
  det = -9
W3 : 3 vectors in R^3, rank = 2, pivot columns = (0, 1)
  DEPENDENT
  det = 0
  dependency (null vector c with Mc = 0): [-1, -1, 1]
W4 : 4 vectors in R^4, rank = 3, pivot columns = (0, 1, 2)
  DEPENDENT
  det = 0
  dependency (null vector c with Mc = 0): [-2, -1, 0, 1]


In [3]:
# Floating-point version: numpy's matrix_rank uses the SVD with a tolerance.
M = np.array([[1, 0, 1, 2], [0, 1, 1, 1], [1, 1, 1, 3], [2, 1, 0, 5]], dtype=float)  # columns = W4 vectors
print("numpy rank:", np.linalg.matrix_rank(M), "  singular values:", np.linalg.svd(M, compute_uv=False))
# A nearly dependent set: rank is 'full' but the smallest singular value warns you (multicollinearity)
N = np.array([[1, 1], [1, 1 + 1e-9]])
print("near-dependent: rank", np.linalg.matrix_rank(N), " singular values", np.linalg.svd(N, compute_uv=False),
      " cond =", f"{np.linalg.cond(N):.2e}")

numpy rank: 3   singular values: [6.9151 1.5477 0.8864 0.    ]
near-dependent: rank 2  singular values [2. 0.]  cond = 4.00e+09


## Part 2: Basis of a span from the pivot columns (W5)

In [4]:
vs = [(1, 2, 1, 0), (2, 4, 2, 0), (0, 1, 1, 1), (1, 3, 2, 1), (1, 1, 1, 1)]
M = cols(*vs)
R, piv = M.rref()
sp.pprint(R)
print("pivot columns:", piv, "-> basis = original vectors", [vs[i] for i in piv], " dim =", len(piv))
print("v2 = 2 v1:", sp.Matrix(vs[1]) == 2 * sp.Matrix(vs[0]))
print("v4 = v1 + v3:", sp.Matrix(vs[3]) == sp.Matrix(vs[0]) + sp.Matrix(vs[2]))

⎡1  2  0  1  0⎤
⎢             ⎥
⎢0  0  1  1  0⎥
⎢             ⎥
⎢0  0  0  0  1⎥
⎢             ⎥
⎣0  0  0  0  0⎦
pivot columns: (0, 2, 4) -> basis = original vectors [(1, 2, 1, 0), (0, 1, 1, 1), (1, 1, 1, 1)]  dim = 3
v2 = 2 v1: True
v4 = v1 + v3: True


## Part 3: Extending an independent set to a basis of ℝ³ (W8)

In [5]:
def extend_to_basis(*vs):
    n = len(vs[0])
    E = [tuple(int(i == j) for j in range(n)) for i in range(n)]   # standard basis
    allv = list(vs) + E
    _, piv = cols(*allv).rref()
    return [allv[i] for i in piv]

B = extend_to_basis((1, 1, 0), (1, 0, 1))
print("extended basis:", B, "  det =", cols(*B).det())
B = extend_to_basis((1, 2, 3))
print("extended basis:", B, "  det =", cols(*B).det())

extended basis: [(1, 1, 0), (1, 0, 1), (1, 0, 0)]   det = 1
extended basis: [(1, 2, 3), (1, 0, 0), (0, 1, 0)]   det = 3


## Part 4: Coordinates and change of basis (W6, W7)

In [6]:
# W6: coordinates of x = (6,5,7) in B = {(1,1,0),(0,1,1),(1,0,1)}
PB = cols((1, 1, 0), (0, 1, 1), (1, 0, 1))
x = sp.Matrix([6, 5, 7])
print("[x]_B =", list(PB.solve(x)), "   det P_B =", PB.det())

# 'Spectacles' example from class: (1,0) in the basis {(2,5),(1,5)}
PB = cols((2, 5), (1, 5))
print("[(1,0)]_B =", list(PB.inv() * sp.Matrix([1, 0])))

[x]_B = [2, 3, 4]    det P_B = 2
[(1,0)]_B = [1, -1]


In [7]:
# W7: B = {(2,5),(1,5)},  C = {(1,1),(1,-1)}
PB = cols((2, 5), (1, 5)); PC = cols((1, 1), (1, -1))
P_CB = PC.inv() * PB          # converts B-coordinates into C-coordinates
P_BC = P_CB.inv()             # converts C-coordinates back into B-coordinates
print("P_{C<-B} ="); sp.pprint(P_CB)
print("P_{B<-C} ="); sp.pprint(P_BC)
xB = sp.Matrix([3, -2])
x = PB * xB
xC = P_CB * xB
print("x (standard) =", list(x), "  [x]_C =", list(xC), "  check P_C [x]_C =", list(PC * xC))
print("back to B:", list(P_BC * xC))

P_{C<-B} =
⎡7/2   3 ⎤
⎢        ⎥
⎣-3/2  -2⎦
P_{B<-C} =
⎡4/5   6/5 ⎤
⎢          ⎥
⎣-3/5  -7/5⎦
x (standard) = [4, 5]   [x]_C = [9/2, -1/2]   check P_C [x]_C = [4, 5]
back to B: [3, -2]


## Part 5: Dimension formula (W10)

In [8]:
def dim_sum_and_intersection(U, W):
    S = sp.Matrix.hstack(U, W)
    dU, dW, dS = U.rank(), W.rank(), S.rank()
    # intersection: U a = W b  <=>  [U | -W] (a, b) = 0
    K = sp.Matrix.hstack(U, -W).nullspace()
    inter = [list(U * k[:U.cols, :]) for k in K]
    print(f"dim U = {dU}, dim W = {dW}, dim(U+W) = {dS}, dim(U∩W) = {dU + dW - dS}; U∩W spanned by {inter}")

dim_sum_and_intersection(cols((1, 0, 0), (0, 1, 0)), cols((1, 1, 0), (0, 0, 1)))          # W10, planes in R^3
dim_sum_and_intersection(cols((1, 1, 0, 0), (0, 1, 1, 0)), cols((0, 0, 1, 1), (1, 2, 1, 0)))  # P22, planes in R^4

dim U = 2, dim W = 2, dim(U+W) = 3, dim(U∩W) = 1; U∩W spanned by [[1, 1, 0]]
dim U = 2, dim W = 2, dim(U+W) = 3, dim(U∩W) = 1; U∩W spanned by [[1, 2, 1, 0]]


## Part 6: Polynomials as coordinate vectors (W9, P18)

In [9]:
t = sp.symbols('x')
# coefficient vectors (const, x, x^2)
Bp = cols((1, 0, 0), (1, 1, 0), (1, 2, 1))           # 1, 1+x, (1+x)^2
c = Bp.solve(sp.Matrix([5, 4, 1]))                   # p = 5 + 4x + x^2
print("coords of 5+4x+x^2 in {1, 1+x, (1+x)^2}:", list(c),
      " check:", sp.expand(c[0] + c[1]*(1 + t) + c[2]*(1 + t)**2))
Bq = cols((1, 1, 0), (1, -1, 0), (0, 0, 1))          # 1+x, 1-x, x^2
print("det =", Bq.det(), " coords of 3+5x-2x^2:", list(Bq.solve(sp.Matrix([3, 5, -2]))))

coords of 5+4x+x^2 in {1, 1+x, (1+x)^2}: [2, 2, 1]  check: x**2 + 4*x + 5
det = -2  coords of 3+5x-2x^2: [4, -1, -2]


## Part 7: Case study, RGB → YCbCr (JPEG/JFIF) is a change of basis

JFIF uses the ITU-R BT.601 weights. Up to the +128 offset on Cb and Cr, it is a 3×3 invertible matrix, so it is a change of basis of colour space ℝ³.

In [10]:
T = np.array([[ 0.299,     0.587,     0.114   ],
              [-0.168736, -0.331264,  0.5     ],
              [ 0.5,      -0.418688, -0.081312]])
print("det T =", round(np.linalg.det(T), 4), "-> invertible, rows/cols form a basis")
print("T^-1 (YCbCr -> RGB) =\n", np.linalg.inv(T))
for name, rgb in [("white", [255, 255, 255]), ("pure red", [255, 0, 0]), ("mid grey", [128, 128, 128])]:
    ycc = T @ np.array(rgb) + np.array([0, 128, 128])
    print(f"{name:9s} RGB {rgb} -> YCbCr {ycc.round(2)}")
# 4:2:0 subsampling keeps Y at full resolution, Cb and Cr at 1/4 each
print("samples per pixel: RGB = 3, YCbCr 4:2:0 =", 1 + 0.25 + 0.25)

det T = 0.2363 -> invertible, rows/cols form a basis
T^-1 (YCbCr -> RGB) =
 [[ 1.     -0.      1.402 ]
 [ 1.     -0.3441 -0.7141]
 [ 1.      1.772   0.    ]]
white     RGB [255, 255, 255] -> YCbCr [255. 128. 128.]
pure red  RGB [255, 0, 0] -> YCbCr [ 76.24  84.97 255.5 ]
mid grey  RGB [128, 128, 128] -> YCbCr [128. 128. 128.]
samples per pixel: RGB = 3, YCbCr 4:2:0 = 1.5


## Part 8: Case study, DCT basis compression on sklearn digits

Each 8×8 digit is a vector in ℝ⁶⁴. The standard basis is 'one pixel at a time'. The orthonormal 2-D DCT-II basis (the JPEG basis) is 64 cosine patterns. Keep only the *k* largest-magnitude coefficients in each basis and compare the reconstruction error.

In [11]:
digits = load_digits()
imgs = digits.images            # (1797, 8, 8), values 0..16
print(imgs.shape)

def keep_top_k(coeffs, k):
    flat = coeffs.reshape(len(coeffs), -1).copy()
    idx = np.argsort(-np.abs(flat), axis=1)[:, k:]
    np.put_along_axis(flat, idx, 0.0, axis=1)
    return flat.reshape(coeffs.shape)

def rel_err(a, b):
    return np.linalg.norm((a - b).reshape(len(a), -1), axis=1) / np.linalg.norm(a.reshape(len(a), -1), axis=1)

D = dctn(imgs, type=2, norm='ortho', axes=(1, 2))
print("orthonormal: energy preserved?", np.allclose((imgs**2).sum(), (D**2).sum()))
print(" k | mean rel. error, pixel basis | mean rel. error, DCT basis")
for k in [4, 8, 16, 32]:
    e_pix = rel_err(imgs, keep_top_k(imgs, k)).mean()
    e_dct = rel_err(imgs, idctn(keep_top_k(D, k), type=2, norm='ortho', axes=(1, 2))).mean()
    print(f"{k:2d} | {e_pix:.3f}                        | {e_dct:.3f}")

(1797, 8, 8)
orthonormal: energy preserved? True
 k | mean rel. error, pixel basis | mean rel. error, DCT basis
 4 | 0.856                        | 0.482
 8 | 0.701                        | 0.381
16 | 0.394                        | 0.262
32 | 0.020                        | 0.124


## Part 9: Case study, PCA = choosing a better basis

In [12]:
rng = np.random.default_rng(0)
X = rng.multivariate_normal([0, 0], [[3, 2], [2, 2]], size=500)
pca = PCA(n_components=2).fit(X)
Q = pca.components_.T                       # columns = new orthonormal basis
print("PCA basis vectors (columns):\n", Q)
print("orthonormal? Q^T Q =\n", Q.T @ Q)
print("explained variance ratio:", pca.explained_variance_ratio_)
Z = (X - X.mean(0)) @ Q                      # coordinates in the new basis
print("covariance of new coordinates (diagonal!):\n", np.cov(Z.T))

PCA basis vectors (columns):
 [[ 0.7896 -0.6137]
 [ 0.6137  0.7896]]
orthonormal? Q^T Q =
 [[1. 0.]
 [0. 1.]]
explained variance ratio: [0.915 0.085]
covariance of new coordinates (diagonal!):
 [[ 4.4289 -0.    ]
 [-0.      0.4112]]


In [13]:
pca64 = PCA().fit(digits.data)
cum = np.cumsum(pca64.explained_variance_ratio_)
for thr in [0.80, 0.90, 0.95, 0.99]:
    print(f"{thr:.0%} of variance needs {np.searchsorted(cum, thr) + 1} of 64 basis vectors")
print("rank of the 1797 x 64 data matrix:", np.linalg.matrix_rank(digits.data))

80% of variance needs 13 of 64 basis vectors
90% of variance needs 21 of 64 basis vectors
95% of variance needs 29 of 64 basis vectors
99% of variance needs 41 of 64 basis vectors
rank of the 1797 x 64 data matrix: 61


## Part 10: Case study, one-hot vs dense embedding bases

In [14]:
vocab = ["king", "queen", "man", "woman", "apple", "banana"]
OH = np.eye(len(vocab))                     # one-hot: standard basis of R^6
print("one-hot rank:", np.linalg.matrix_rank(OH), " all pairwise dot products zero:",
      np.allclose(OH @ OH.T - np.eye(6), 0))
# toy dense 3-D embeddings (hand-made: royalty, gender, fruit)
E = np.array([[0.9, 0.8, 0.0], [0.9, -0.8, 0.0], [0.1, 0.8, 0.0],
              [0.1, -0.8, 0.0], [0.0, 0.0, 0.9], [0.0, 0.1, 0.8]])
print("dense: 6 vectors in R^3 -> rank", np.linalg.matrix_rank(E), "(must be dependent: 6 > 3)")
print("king - man + woman =", E[0] - E[2] + E[3], " queen =", E[1])

one-hot rank: 6  all pairwise dot products zero: True
dense: 6 vectors in R^3 -> rank 3 (must be dependent: 6 > 3)
king - man + woman = [ 0.9 -0.8  0. ]  queen = [ 0.9 -0.8  0. ]


## Part 11: Case study, the dummy-variable trap

One-hot encoding a categorical feature with *all* its categories **and** keeping an intercept column makes the design matrix's columns linearly dependent: the one-hot columns add up to the all-ones column. Then XᵀX is singular and the regression weights are not unique.

In [15]:
rng = np.random.default_rng(1)
city = rng.integers(0, 3, size=200)                 # 3 cities
area = rng.uniform(50, 150, size=200)
onehot = np.eye(3)[city]
X_trap = np.c_[np.ones(200), area, onehot]           # intercept + area + 3 dummies = 5 columns
X_ok   = np.c_[np.ones(200), area, onehot[:, 1:]]    # drop one dummy (reference category)
for name, X in [("intercept + all 3 dummies", X_trap), ("intercept + 2 dummies   ", X_ok)]:
    r = np.linalg.matrix_rank(X)
    print(f"{name}: {X.shape[1]} columns, rank {r}, cond(X^T X) = {np.linalg.cond(X.T @ X):.2e}")
print("dependency: d1 + d2 + d3 - 1 =", np.abs(onehot.sum(1) - 1).max())
# two different weight vectors give identical predictions in the trap
w = np.array([10., 2., 1., 2., 3.])
w2 = w + 5 * np.array([1, 0, -1, -1, -1])           # move along the null-space direction
print("same predictions with different weights:", np.allclose(X_trap @ w, X_trap @ w2))

intercept + all 3 dummies: 5 columns, rank 4, cond(X^T X) = 2.65e+18
intercept + 2 dummies   : 4 columns, rank 4, cond(X^T X) = 1.80e+05
dependency: d1 + d2 + d3 - 1 = 0.0
same predictions with different weights: True


## Part 12: Practice-problem answer checks

In [16]:
C = cols
print("P2 :", list(C((1, 1), (1, -1)).solve(sp.Matrix([7, 4]))))
print("P3 : det =", C((1, 0, 1), (0, 1, 1), (1, 1, 0)).det(), "  P4 : det =", C((1, 2, 3), (4, 5, 6), (7, 8, 9)).det())
k = sp.symbols('k'); print("P6 : k =", sp.solve(C((1, k), (k, 4)).det(), k))
print("P8 : dets", [C(*s).det() for s in [((1, 2, 3), (0, 1, 2), (0, 0, 1)), ((1, 1, 1), (2, 2, 2), (0, 0, 1))]])
print("P11:", list(C((1, 0, 0), (1, 1, 0), (1, 1, 1)).solve(sp.Matrix([1, 2, 3]))))
PB, PC = C((1, 2), (3, 5)), C((1, 0), (1, 1)); P = PC.inv() * PB
print("P12: P_{C<-B} =", P.tolist(), " P_{B<-C} =", P.inv().tolist(), " [x]_C =", list(P * sp.Matrix([2, -1])))
R, piv = C((1, 2, 0, 1), (2, 4, 1, 3), (3, 6, 1, 4), (0, 0, 1, 1)).rref(); print("P13: pivots", piv); sp.pprint(R)
print("P14:", extend_to_basis((1, 0, 1, 0), (0, 1, 0, 1)))
a = sp.symbols('a'); print("P15: det =", sp.factor(C((1, 1, 1), (1, a, 1), (1, 1, a)).det()))
print("P16: det of coefficient matrix =", sp.Matrix([[1, 0, 1], [1, 1, 0], [0, 1, 1]]).det())
print("P18:", list(C((1, 1, 0), (1, -1, 0), (0, 0, 1)).solve(sp.Matrix([3, 5, -2]))))
print("P20: rank of {x-1, x^2-1, x^3-1} =", C((-1, 1, 0, 0), (-1, 0, 1, 0), (-1, 0, 0, 1)).rank())
A = C((1, 1, 2), (1, -1, 1))
for b in [(3, 1, 5), (3, 1, 4)]:      # worked example in Section 5
    print("S5 :", b, "in span?", sp.Matrix.hstack(A, sp.Matrix(b)).rank() == A.rank())
h = sp.symbols('h')
x12 = sp.solve([sp.Eq(sp.Symbol('x1') + sp.Symbol('x2'), 2), sp.Eq(sp.Symbol('x1') - sp.Symbol('x2'), 0)])
print("P21: x =", x12, " -> h = 2*x1 + x2 =", 2 * x12[sp.Symbol('x1')] + x12[sp.Symbol('x2')],
      " rank check:", sp.Matrix.hstack(A, sp.Matrix([2, 0, 3])).rank() == A.rank())
X = sp.Matrix([[1, 1, 0, 1], [1, 2, 1, 3], [1, 0, 2, 2], [1, 3, 1, 4], [1, 1, 1, 2]])
print("P24: rank", X.rank(), " det(X^T X) =", (X.T * X).det(), " null vector", list(X.nullspace()[0]))
print("P25: dim =", 4 - sp.Matrix([[1, 1, 1, 1], [1, -1, 0, 0]]).rank())
q1 = sp.Matrix([1, 1]) / sp.sqrt(2); q2 = sp.Matrix([-1, 1]) / sp.sqrt(2); v = sp.Matrix([2, 1])
print("P26:", sp.nsimplify(q1.dot(v)), sp.nsimplify(q2.dot(v)), " check:", list(sp.simplify(q1 * q1.dot(v) + q2 * q2.dot(v))))

P2 : [11/2, 3/2]
P3 : det = -2   P4 : det = 0
P6 : k = [-2, 2]
P8 : dets [1, 0]
P11: [-1, -1, 3]
P12: P_{C<-B} = [[-1, -2], [2, 5]]  P_{B<-C} = [[-5, -2], [2, 1]]  [x]_C = [0, -1]
P13: pivots (0, 1)
⎡1  0  1  -2⎤
⎢           ⎥
⎢0  1  1  1 ⎥
⎢           ⎥
⎢0  0  0  0 ⎥
⎢           ⎥
⎣0  0  0  0 ⎦
P14: [(1, 0, 1, 0), (0, 1, 0, 1), (1, 0, 0, 0), (0, 1, 0, 0)]
P15: det = (a - 1)**2
P16: det of coefficient matrix = 2
P18: [4, -1, -2]
P20: rank of {x-1, x^2-1, x^3-1} = 3
S5 : (3, 1, 5) in span? True
S5 : (3, 1, 4) in span? False
P21: x = {x1: 1, x2: 1}  -> h = 2*x1 + x2 = 3  rank check: True
P24: rank 3  det(X^T X) = 0  null vector [0, -1, -1, 1]
P25: dim = 2
P26: 3*sqrt(2)/2 -sqrt(2)/2  check: [2, 1]
